### Fit T2R traces for Q4_2p90

In [ ]:
from pathlib import Path

from betata.qubit_measurements.qubit import Qubit, load_qubit
from betata.qubit_measurements.traces import T2RTrace, load_t2r_traces, save_t2r_results
from betata.qubit_measurements.fit_t2r_traces.fit_t2r_traces import fit_t2r_trace, plot_t2r_trace, plot_t2r_vs_time

CWD = Path.cwd()

Specify input folder and output file

In [ ]:
qubit_name = "Q4_2p90"
input_folder = CWD / f"data/qubit_measurements/{qubit_name}/T2R_{qubit_name}"
output_folder = CWD / f"out/qubit_measurements/{qubit_name}/T2R_{qubit_name}"
qubit_file = CWD / f"out/qubit_measurements/{qubit_name}.h5"

Load qubit

In [ ]:
qubit: Qubit = load_qubit(qubit_file)

Load traces

In [ ]:
traces: list[T2RTrace] = load_t2r_traces(input_folder)

Fit traces and save plots

In [ ]:
fit_results = {}
for trace in traces:
    fit_result = fit_t2r_trace(trace, save_folder=output_folder)
    fit_results[trace.id] = fit_result

Inspect and exclude certain traces

In [ ]:
traces_to_inspect = []
for trace in traces:
    if trace.id in traces_to_inspect:
        plot_t2r_trace(trace, fit_params=fit_results[trace.id].params)

In [ ]:
bad_fits_to_exclude = [
    8, 9, 11, 14, 15, 17, 25, 30, 36, 39, 43, 44, 45, 46, 48, 49, 61, 65, 69, 76, 84, 89, 97, 101, 123, 151, 154, 156, 157, 159, 184, 186, 188, 198, 199, 200, 205, 217, 218, 219, 220, 221, 248, 258, 260, 261, 269, 270, 273, 275, 278, 280, 281, 293, 310, 320, 331, 345, 347, 348, 350, 352, 353, 355, 359, 376, 385, 386, 387, 389, 392, 399, 402, 407, 409, 417, 422, 426, 436, 442, 460, 463, 464, 487, 542, 545, 550, 568, 579, 583, 584, 587, 616, 626, 629, 631, 655, 697, 699, 701, 712, 722, 727, 730, 735, 737, 739, 741, 745, 765,
]

for trace in traces:
    if trace.id in bad_fits_to_exclude:
        trace.is_excluded = True
    else:
        trace.is_excluded = False

    if None in [trace.T2R, trace.T2R_err] or trace.T2R_err / trace.T2R > 0.5:
        trace.is_excluded = True 

included_traces = [tr for tr in traces if not tr.is_excluded]

In [ ]:
t2e_vs_time_fig = plot_t2r_vs_time(included_traces, qubit.name)

In [ ]:
for trace in included_traces:
    if trace.T2R > 300e-6:
        plot_t2r_trace(trace, fit_params=fit_results[trace.id].params)

Save fit results to qubit file

In [ ]:
save_t2r_results(included_traces, qubit)